# 134 - DeerFlow Embedded Client

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/134-deerflow-embedded-client/deerflow_embedded_client_workbook.ipynb)

**What you will learn:**

- DeerFlow's three-endpoint HTTP API (upload / stream SSE / blocking chat)
- How Server-Sent Events (SSE) work at the protocol level: `data:` lines and the `[DONE]` sentinel
- How to implement a `DeerFlowClient` class from scratch
- What `plan_mode=True` adds: a visible planning step before execution
- Error handling: connection refused, stream interruption, and malformed JSON
- How DeerFlow's ownership model differs from LangGraph, Google ADK, and Agno

**Source code:** `examples/134-deerflow-embedded-client/`

**Two halves:**
- **Part A** - CPU-safe concept demonstrations (no server required)
- **Part B** - requires a running DeerFlow server at `localhost:8000`

In [ ]:
%pip install -q httpx python-dotenv

## Part A - CPU-Safe Concept Demonstrations

### A1 - DeerFlow Architecture: What It Owns vs What You Own

| Approach | How you drive it | What the runtime owns |
|---|---|---|
| LangGraph | `app.invoke(state)` | Nothing (you build everything) |
| Google ADK | `runner.run_async()` | Tool-call loop |
| Agno | `agent.run()` | Memory and tool registry |
| DeerFlow client | `client.stream()` -> events | Skills, memory, tools, subagents |

DeerFlow is a full research agent stack. When you call `stream()`, you are sending a message
into DeerFlow's LLM loop. DeerFlow owns all the tools (web search, file read, code execution),
the memory, and the skill dispatcher. You own only the HTTP request and the event parsing.

This is a fundamentally different ownership boundary from frameworks like LangGraph, where you
build the entire graph, register the tools, and manage all state yourself. With DeerFlow, the
research pipeline already exists; you are a client calling into it.

### A2 - The Three DeerFlow API Endpoints

#### 1. Upload a file

```
POST /api/files/upload
Content-Type: multipart/form-data

file: <filename, bytes, mime_type>
thread_id: <string>

Response: { "artifact_id": "<string>" }
```

Use when you want DeerFlow to read a document you provide. The returned `artifact_id` is
referenced by DeerFlow internally; you do not need to include it in subsequent chat messages.

#### 2. Stream a chat response (SSE)

```
POST /api/chat/stream
Content-Type: application/json

{
  "message": "<string>",
  "thread_id": "<string>",
  "plan_mode": false,
  "subagent_enabled": false
}

Response: text/event-stream (SSE)
```

Use for interactive sessions where you want to show progress as DeerFlow works.
Each SSE event carries a JSON payload with a `type` field.

#### 3. Blocking chat

```
POST /api/chat
Content-Type: application/json

{ "message": "<string>", "thread_id": "<string>" }

Response: { "content": "<string>" }
```

Use when you do not need streaming and just want the final text. Simpler but gives no
progress visibility.

**Note:** `thread_id` ties all three endpoints together. Uploading a file and then streaming
a question on the same `thread_id` means DeerFlow can reference the uploaded document.

### A3 - Server-Sent Events (SSE) Protocol

SSE is a simple HTTP long-poll format where the server pushes newline-delimited `data:` lines:

```
HTTP/1.1 200 OK
Content-Type: text/event-stream

data: {"type": "message_chunk", "content": "Hello"}

data: {"type": "tool_call", "tool_name": "web_search"}

data: [DONE]

```

**Parse rules:**
1. Skip any line that does not start with `data:`
2. Strip the `data: ` prefix from the line
3. If the remainder is `[DONE]`, the stream is finished - stop iterating
4. Otherwise, `json.loads()` the remainder to get the event dict
5. Read `event["type"]` to decide what to do with the payload

**Connection lifecycle:** The HTTP connection stays open as a long-poll while the server
pushes events. No special client library is needed - `httpx`'s `stream()` context manager
and `iter_lines()` are sufficient. The connection closes naturally after the `[DONE]` sentinel.

In [ ]:
# Part A: DeerFlowClient class definition
# Self-contained -- no server required to define or inspect this class.

import json
from dataclasses import dataclass, field
from typing import Iterator

import httpx


@dataclass
class DeerFlowClient:
    """HTTP client for the DeerFlow research agent API."""

    base_url: str
    thread_id: str

    # field(default_factory=...) is required here because httpx.Client is mutable.
    # A plain default= would share one client instance across all DeerFlowClient instances.
    _http: httpx.Client = field(
        default_factory=lambda: httpx.Client(
            timeout=httpx.Timeout(60.0, read=120.0)
        )
    )

    def upload(self, filename: str, content: str) -> str:
        """Upload a text document to the current thread.

        Returns the artifact_id assigned by DeerFlow.
        DeerFlow can reference the file in subsequent chat turns on the same thread_id.
        """
        resp = self._http.post(
            f"{self.base_url}/api/files/upload",
            files={"file": (filename, content.encode(), "text/markdown")},
            data={"thread_id": self.thread_id},
        )
        resp.raise_for_status()
        return resp.json().get("artifact_id", "unknown")

    def stream(
        self,
        message: str,
        *,
        plan_mode: bool = False,
        subagent_enabled: bool = False,
    ) -> Iterator[tuple]:
        """Stream events from DeerFlow as a generator of (event_type, data) tuples.

        Yields one tuple per SSE event until [DONE] is received.
        The generator pattern lets callers process events one at a time
        without buffering the entire response in memory.
        """
        with self._http.stream(
            "POST",
            f"{self.base_url}/api/chat/stream",
            json={
                "message": message,
                "thread_id": self.thread_id,
                "plan_mode": plan_mode,
                "subagent_enabled": subagent_enabled,
            },
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                raw = line.removeprefix("data:").strip()
                if raw == "[DONE]":
                    yield "end", {}
                    return
                try:
                    ev = json.loads(raw)
                    yield ev.get("type", "unknown"), ev
                except json.JSONDecodeError:
                    # Malformed lines are skipped -- production SSE streams
                    # occasionally emit keep-alive or comment lines.
                    pass

    def chat(self, message: str, *, plan_mode: bool = False) -> str:
        """Blocking convenience wrapper -- returns the full text response.

        Joins all message_chunk content values from the stream.
        Use this when you do not need streaming progress updates.
        """
        return "".join(
            d.get("content", "")
            for et, d in self.stream(message, plan_mode=plan_mode)
            if et == "message_chunk"
        )


print("DeerFlowClient defined.")
print("Methods:", [m for m in dir(DeerFlowClient) if not m.startswith("_")])

In [ ]:
# Part A: SSE parser demo on a mock stream
# Exercises the same parsing logic used inside DeerFlowClient.stream().

MOCK_SSE_LINES = [
    "data: {\"type\": \"tool_call\", \"tool_name\": \"web_search\", \"args\": {\"query\": \"DeerFlow architecture\"}}",
    "data: {\"type\": \"tool_result\", \"content\": \"DeerFlow is a ByteDance research agent framework.\"}",
    "data: {\"type\": \"message_chunk\", \"content\": \"DeerFlow is a \"}",
    "data: {\"type\": \"message_chunk\", \"content\": \"full research agent stack.\"}",
    "data: {\"type\": \"end\"}",
    "data: [DONE]",
]

ICONS = {
    "tool_call": "[tool]",
    "tool_result": "[result]",
    "message_chunk": "[chunk]",
    "end": "[end]",
    "unknown": "[?]",
}


def parse_sse_stream(lines):
    """Parse a list of raw SSE lines into a list of (event_type, data) tuples."""
    events = []
    for line in lines:
        if not line.startswith("data:"):
            continue
        raw = line.removeprefix("data:").strip()
        if raw == "[DONE]":
            events.append(("end", {}))
            break
        try:
            ev = json.loads(raw)
            events.append((ev.get("type", "unknown"), ev))
        except json.JSONDecodeError:
            pass
    return events


parsed = parse_sse_stream(MOCK_SSE_LINES)
for event_type, data in parsed:
    icon = ICONS.get(event_type, "[?]")
    detail = data.get("content", data.get("tool_name", ""))
    print(f"  {icon}  {event_type:15s}  {detail!r}")

print(f"\nParsed {len(parsed)} events total.")

### A4 - SSE Event Types Reference

| Event Type | When It Appears | Key Data Fields | Icon |
|---|---|---|---|
| `message_chunk` | LLM generating a token | `content: str` | `[chunk]` |
| `tool_call` | Agent invoking a tool | `tool_name: str`, `args: dict` | `[tool]` |
| `tool_result` | Tool execution complete | `content: str` | `[result]` |
| `end` | Run finished (no content) | `{}` | `[end]` |

**Important distinction:**

`[DONE]` is the SSE *transport* sentinel - it is a raw string in the `data:` line, not JSON.
It signals that the HTTP long-poll connection is closing. It is separate from the `"end"` event
type, which is a JSON payload indicating the agent finished its run.

In `DeerFlowClient.stream()`, receiving `[DONE]` causes the generator to yield `("end", {})`
and return, collapsing both signals into one.

In [ ]:
# Part A: plan_mode comparison demo
# Shows the difference in event sequences with and without plan_mode=True.

MOCK_NO_PLAN = [
    'data: {"type": "message_chunk", "content": "Paris is the capital of France."}',
    'data: {"type": "message_chunk", "content": " It is known for the Eiffel Tower."}',
    'data: {"type": "end"}',
    'data: [DONE]',
]

MOCK_WITH_PLAN = [
    'data: {"type": "tool_call", "tool_name": "plan", "args": {"steps": ["recall capital from knowledge", "add landmark detail"]}}',
    'data: {"type": "tool_result", "content": "Plan accepted."}',
    'data: {"type": "message_chunk", "content": "Paris is the capital of France."}',
    'data: {"type": "message_chunk", "content": " It is known for the Eiffel Tower."}',
    'data: {"type": "end"}',
    'data: [DONE]',
]

no_plan_events = parse_sse_stream(MOCK_NO_PLAN)
with_plan_events = parse_sse_stream(MOCK_WITH_PLAN)

print("plan_mode=False  event sequence:")
for et, _ in no_plan_events:
    print(f"  {et}")

print()
print("plan_mode=True   event sequence:")
for et, data in with_plan_events:
    extra = ""
    if et == "tool_call":
        extra = f"  <- steps: {data.get('args', {}).get('steps')}"
    print(f"  {et}{extra}")

print()
print("With plan_mode=True you see what DeerFlow intends to do before it starts.")
print("This is useful for debugging skill routing and building progress UIs.")

In [ ]:
# Part A: HTTP request shapes -- no server needed
# Prints what each request would look like on the wire.

import json as _json

print("GET /api/health")
print("  No body")
print()

print("POST /api/files/upload")
print("  Content-Type: multipart/form-data")
print("  file: notes.md (text/markdown)")
print("  thread_id: demo-thread-001")
print("  Response: { 'artifact_id': '<uuid>' }")
print()

chat_body = {
    "message": "Summarise the notes",
    "thread_id": "demo-thread-001",
    "plan_mode": True,
    "subagent_enabled": False,
}
print("POST /api/chat/stream")
print("  Content-Type: application/json")
print(f"  Body: {_json.dumps(chat_body, indent=4)}")
print("  Response: text/event-stream (SSE)")
print()

blocking_body = {
    "message": "What is the main finding?",
    "thread_id": "demo-thread-001",
}
print("POST /api/chat")
print("  Content-Type: application/json")
print(f"  Body: {_json.dumps(blocking_body, indent=4)}")
print("  Response: { 'content': '<string>' }")

### A5 - Error Handling Patterns

#### Scenario 1: Connection refused

```python
try:
    events = list(client.stream("Hello"))
except httpx.ConnectError as exc:
    raise RuntimeError(
        "DeerFlow unreachable. Start it with `make dev` in the deer-flow repo."
    ) from exc
```

#### Scenario 2: Stream interrupted mid-way

If the connection drops while `iter_lines()` is running, `httpx` raises
`httpx.RemoteProtocolError` or `httpx.ReadError`. Events yielded before the error are already
processed - only the remainder is lost. Wrap the iteration in a try/except if partial results
are acceptable.

#### Scenario 3: Malformed SSE JSON

The `json.JSONDecodeError` is silently caught with `pass` inside `stream()`. This is
intentional: SSE streams sometimes include keep-alive comments or blank lines that are not
JSON. Skipping them is the correct behavior.

**Never use a bare `except:` clause** - always catch specific exception classes.

| Error | Exception class | Recommended action |
|---|---|---|
| Server not running | `httpx.ConnectError` | Raise with setup instructions |
| Connection dropped mid-stream | `httpx.ReadError` | Log and use partial results |
| Malformed SSE line | `json.JSONDecodeError` | Skip silently |
| Non-2xx HTTP status | `httpx.HTTPStatusError` | Raise immediately via `raise_for_status()` |

In [ ]:
# Part A: error handling demo

# Scenario 1: malformed JSON in SSE stream
malformed_lines = [
    'data: {"type": "message_chunk", "content": "Hello"}',
    'data: {broken json here',
    'data: {"type": "message_chunk", "content": " world"}',
    'data: [DONE]',
]

events = parse_sse_stream(malformed_lines)
print("Malformed JSON handling:")
print(f"  Input lines:   {len(malformed_lines)}")
print(f"  Parsed events: {len(events)} (bad line silently skipped)")
text = "".join(d.get("content", "") for et, d in events if et == "message_chunk")
print(f"  Recovered text: {text!r}")
print()

# Scenario 2: retry wrapper pattern
def stream_with_retry(client_stream_fn, message, max_retries=1):
    """Retry once on ConnectError -- useful for flaky local dev servers."""
    for attempt in range(max_retries + 1):
        try:
            return list(client_stream_fn(message))
        except Exception as exc:
            if attempt < max_retries:
                print(f"  Attempt {attempt + 1} failed: {exc}. Retrying...")
            else:
                raise
    return []


print("Retry wrapper defined. See Exercise 1 for the full implementation.")

### Exercise 1 - Add Retry Logic to DeerFlowClient

**Problem:** Add a `retry_stream()` method to `DeerFlowClient` that retries once on
`httpx.ConnectError`.

**Requirements:**
- First attempt: call `self.stream()` as normal
- If `ConnectError` is raised: retry once
- If the second attempt also fails: re-raise as a `RuntimeError` with a helpful message
- Return an iterator just like `stream()` (use `yield from`)

**Hint:** Use a `for attempt in range(max_retries + 1)` loop and catch `httpx.ConnectError`
specifically.

The cell below contains the TODO stub. The answer key follows.

In [ ]:
# Exercise 1 -- TODO stub

class DeerFlowClientWithRetry(DeerFlowClient):
    def retry_stream(self, message: str, *, plan_mode: bool = False, max_retries: int = 1):
        """TODO: implement retry logic here.

        Should yield from self.stream() with one retry on httpx.ConnectError.
        On the final failed attempt, raise RuntimeError with a setup hint.
        """
        # Hint:
        # for attempt in range(max_retries + 1):
        #     try:
        #         yield from self.stream(message, plan_mode=plan_mode)
        #         return
        #     except httpx.ConnectError as exc:
        #         if attempt < max_retries: print("retrying...")
        #         else: raise RuntimeError("...") from exc
        pass


print("TODO: implement retry_stream() above, then test it.")
print("When complete, the mock test in the answer key cell should print: Exercise 1 passed.")

In [ ]:
# Exercise 1 -- Answer Key


class DeerFlowClientWithRetry(DeerFlowClient):
    def retry_stream(
        self,
        message: str,
        *,
        plan_mode: bool = False,
        max_retries: int = 1,
    ) -> Iterator:
        """Stream with one retry on ConnectError."""
        for attempt in range(max_retries + 1):
            try:
                yield from self.stream(message, plan_mode=plan_mode)
                return  # success -- exit the retry loop
            except httpx.ConnectError as exc:
                if attempt < max_retries:
                    print(f"  Connection failed (attempt {attempt + 1}), retrying...")
                else:
                    raise RuntimeError(
                        "DeerFlow unreachable after retries. "
                        "Start it with `make dev` in the deer-flow repo."
                    ) from exc


# Mock test (no real server needed)
_call_count = 0


def _mock_stream_fails_once(self, message, *, plan_mode=False, subagent_enabled=False):
    """Simulates a server that is down on the first call, then up on the second."""
    global _call_count
    _call_count += 1
    if _call_count == 1:
        raise httpx.ConnectError("Connection refused (mock)")
    yield "message_chunk", {"content": "Retry succeeded!"}
    yield "end", {}


# Temporarily patch the stream method for testing
_original_stream = DeerFlowClientWithRetry.stream
DeerFlowClientWithRetry.stream = _mock_stream_fails_once

client_retry = DeerFlowClientWithRetry(base_url="http://localhost:8000", thread_id="test-001")
result = list(client_retry.retry_stream("Hello"))

print("Events received after retry:", result)
assert any(et == "message_chunk" for et, _ in result), "Should have received a message_chunk event"
assert _call_count == 2, "Should have called stream() exactly twice"

DeerFlowClientWithRetry.stream = _original_stream  # restore
print("Exercise 1 passed.")

### Exercise 2 - Extract Structured Output from an Event Stream

**Problem:** Write two extraction functions:

1. `extract_final_answer(events)` - returns the concatenated text from all `message_chunk`
   events, ignoring `tool_call` and `tool_result` events entirely.

2. `extract_plan_steps(events)` - returns a list of plan step strings from `tool_call` events
   where `tool_name == "plan"`. The steps are stored under `data["args"]["steps"]`.

Both functions take a list of `(event_type, data)` tuples as returned by `parse_sse_stream()`.

In [ ]:
# Exercise 2 -- TODO stub

def extract_final_answer(events):
    """Return the concatenated text from all message_chunk events."""
    pass  # TODO


def extract_plan_steps(events):
    """Return list of plan step strings from tool_call events where tool_name=='plan'."""
    pass  # TODO


# Test data
_test_events = [
    ("tool_call", {"tool_name": "plan", "args": {"steps": ["step 1", "step 2"]}}),
    ("tool_result", {"content": "Plan confirmed."}),
    ("message_chunk", {"content": "The answer is "}),
    ("message_chunk", {"content": "42."}),
    ("end", {}),
]

answer = extract_final_answer(_test_events)
steps = extract_plan_steps(_test_events)
print(f"Final answer: {answer!r}")   # expected: 'The answer is 42.'
print(f"Plan steps:   {steps}")       # expected: ['step 1', 'step 2']

In [ ]:
# Exercise 2 -- Answer Key

def extract_final_answer(events):
    """Return the concatenated text from all message_chunk events."""
    return "".join(d.get("content", "") for et, d in events if et == "message_chunk")


def extract_plan_steps(events):
    """Return list of plan step strings from tool_call events where tool_name=='plan'."""
    steps = []
    for et, data in events:
        if et == "tool_call" and data.get("tool_name") == "plan":
            raw_steps = data.get("args", {}).get("steps", [])
            steps.extend(raw_steps)
    return steps


_test_events = [
    ("tool_call", {"tool_name": "plan", "args": {"steps": ["read files", "summarise"]}}),
    ("tool_result", {"content": "Plan confirmed."}),
    ("message_chunk", {"content": "The answer is "}),
    ("message_chunk", {"content": "42."}),
    ("end", {}),
]

answer = extract_final_answer(_test_events)
steps = extract_plan_steps(_test_events)
print(f"Final answer: {answer!r}")
print(f"Plan steps:   {steps}")
assert answer == "The answer is 42."
assert steps == ["read files", "summarise"]
print("Exercise 2 passed.")

## Part B - Live DeerFlow Run

**Requirements:** a running DeerFlow server at `http://localhost:8000`.

**Setup:**

```bash
git clone https://github.com/bytedance/deer-flow ../deer-flow
cd ../deer-flow && pip install -e . && make dev
```

**Optional:** set `DEERFLOW_BASE_URL` in your `.env` if the server runs on a different port:

```bash
export DEERFLOW_BASE_URL=http://localhost:8000
```

Cell c021 performs a fail-fast health check. If DeerFlow is not reachable it raises
immediately with setup instructions - there are no silent hangs.

In [ ]:
# Part B: fail-fast health check

import os
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("DEERFLOW_BASE_URL", "http://localhost:8000")
THREAD_ID = "notebook-thread-134"

try:
    r = httpx.get(f"{BASE_URL}/api/health", timeout=5)
    r.raise_for_status()
    print(f"DeerFlow reachable at {BASE_URL}")
except Exception as exc:
    raise RuntimeError(
        f"DeerFlow is not reachable at {BASE_URL}.\n"
        "Start it with `make dev` in the deer-flow repo.\n"
        "See runtime/README.md for full setup instructions."
    ) from exc

client = DeerFlowClient(base_url=BASE_URL, thread_id=THREAD_ID)
print(f"Client ready  thread_id={THREAD_ID}")

In [ ]:
# Part B: upload a document, then stream a query with plan_mode=True

SAMPLE_CORPUS = (
    "# Agent Framework Patterns\n"
    "\n"
    "## LangGraph\n"
    "LangGraph uses a directed graph where nodes are Python functions and edges are\n"
    "conditional transitions. State is passed through the graph as a typed dict.\n"
    "The developer builds every node and tool from scratch.\n"
    "\n"
    "## Google ADK\n"
    "Google Agent Development Kit provides a tool-call loop out of the box.\n"
    "Developers register tools and let the ADK runner handle the LLM/tool iteration.\n"
    "\n"
    "## Agno\n"
    "Agno manages memory and a tool registry. Agents persist conversation state\n"
    "across sessions and can call registered tools by name.\n"
    "\n"
    "## DeerFlow\n"
    "DeerFlow is a full research pipeline. The client sends a message via HTTP\n"
    "and receives a stream of SSE events. DeerFlow owns skills, memory, tools,\n"
    "and optional subagents internally.\n"
)

print("Uploading sample corpus...")
artifact_id = client.upload("framework_patterns.md", SAMPLE_CORPUS)
print(f"Upload complete  artifact_id={artifact_id}")
print()

QUERY = "Compare the ownership models of the four agent frameworks described in the document."
print(f"Streaming query (plan_mode=True): {QUERY!r}")
print("-" * 60)

live_events = []
for event_type, data in client.stream(QUERY, plan_mode=True):
    live_events.append((event_type, data))
    icon = ICONS.get(event_type, "[?]")
    if event_type == "message_chunk":
        print(data.get("content", ""), end="", flush=True)
    elif event_type == "tool_call":
        print(f"\n{icon} {data.get('tool_name', '')}")
    elif event_type == "tool_result":
        snippet = str(data.get("content", ""))[:80]
        print(f"{icon} {snippet}")
    elif event_type == "end":
        print(f"\n{icon} stream finished")

print("-" * 60)
print(f"Total events received: {len(live_events)}")

In [ ]:
# Part B: blocking chat + event summary

from collections import Counter

FOLLOWUP = "Which framework requires the least boilerplate for a simple research task?"
print(f"Blocking chat: {FOLLOWUP!r}")
print("-" * 60)

answer_text = client.chat(FOLLOWUP)
print(answer_text)
print("-" * 60)
print()

# Event summary from the streaming run above
event_counts = Counter(et for et, _ in live_events)
print("Event summary from streaming run:")
for etype, count in sorted(event_counts.items()):
    print(f"  {etype:20s} {count}")

print()
print(f"Thread ID:  {THREAD_ID}")
print(f"Server URL: {BASE_URL}")

## What's Next

- **SSE spec (MDN):** https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events
  Authoritative reference for the event-stream protocol, including multi-line data fields
  and retry directives.

- **DeerFlow GitHub:** https://github.com/bytedance/deer-flow
  Source repository with config docs, SKILL.md format reference, and the full tool registry.

- **Next example in this repo:** `examples/135-deerflow-research-skill/`
  Register a custom `SKILL.md` and trigger it with `@course-research` in a chat message.

- **Example 99 in this repo:** browser agent pattern - another way to drive research
  workflows, this time with a Playwright-backed tool rather than an HTTP API.